In [ ]:
import pandas as pd
import matplotlib 
import seaborn as sns
import matplotlib.pyplot as plt
from cfg import OTHER_METADATA
from utils_figstyle import set_nature_style


set_nature_style()
FIG_SIZE = (8, 3)


In [ ]:
# read in all the csvs in data_sweep/20250924_best_run
df_list = []

# for each csv in data_sweep/20250924_best_run, read in the csv and append to df_list. at the end, concat them all
csvs = [
    "/srv/marl/sonja/zfish_best_runs/runs_raw_metrics_20250924_093829_20250924_150318_wb.csv",
    "/srv/marl/sonja/zfish_best_runs/runs_raw_metrics_20250924_125804_20250924_135046_lmp.csv",
    "/srv/marl/sonja/zfish_best_runs/runs_raw_metrics_20250924_131031_20250924_140129_ltp.csv",
    "/srv/marl/sonja/zfish_best_runs/runs_raw_metrics_20250924_142306_20250924_151352_vd.csv",
    "/srv/marl/sonja/zfish_best_runs/runs_raw_metrics_20250924_150351_20250924_150413_control.csv",
]

for csv in csvs:
    df = pd.read_csv(csv)
    df_list.append(df)

df_runs = pd.concat(df_list, ignore_index=True)

# df_runs = pd.read_csv("sweep_summary_csvs/runs_raw_metrics_20250923_213738_20250924_060835.csv")
print(df_runs.shape)
print(df_runs.columns)



In [ ]:
df_runs.loc[(df_runs['walkerbots'] == 0) & (df_runs['vd'] == 0.006)]

In [ ]:
# df_runs = pd.read_csv("data_sweep/runs_raw_metrics.csv")

# print(df_runs.head())

df_runs["auc_difference"] = df_runs["avg_auc_success"] - df_runs["avg_auc_nontracking"]
sweep_dimensions = ['vd', 'large_move_penalty', 'large_turn_penalty', 'walkerbots']
metrics = ["eating_events_per_episode", 
        #    "avg_auc_success", 
        #    "avg_auc_nontracking", 
           "auc_difference",]

for c in sweep_dimensions:
    df_runs[c] = pd.to_numeric(df_runs[c], errors="coerce")
    df_runs[c] = df_runs[c].abs()



## Using all seeds, plot mean +/- (std, sem) across levels

In [ ]:
# summary_tables = {}

# for dim in sweep_dimensions:
#     grouped = (
#         df_runs.groupby(dim)[metrics]
#         .agg(['mean', 'std', 'sem', 'count'])
#         .reset_index()
#         .sort_values(dim)
#     )
#     summary_tables[dim] = grouped
#     print(f"\n=== Summary by {dim} ===")
#     print(grouped.to_string(index=False))


# for metric in metrics:
#     fig, axes = plt.subplots(1, 4, figsize=FIG_SIZE, sharey=True)
#     for i, dim in enumerate(sweep_dimensions):
#         sns.pointplot(
#             data=df_runs, x=dim, y=metric, errorbar="sd",
#             ax=axes[i], capsize=0.1
#         )
#         axes[i].set_xlabel(OTHER_METADATA[dim]["name"])
#         axes[i].set_ylabel(OTHER_METADATA[metric]["name"])

#         # axes[i].set_title(f"{metric} vs {dim}")
#     plt.tight_layout()
#     plt.show()


default_values = {
    'vd': 0.006,
    'large_move_penalty': 0.01,
    'large_turn_penalty': 0.01,
    'walkerbots': 3
}

summary_tables = {}

for dim in sweep_dimensions:
    # Filter to only include runs with default values for non-swept parameters
    filtered_df = df_runs.copy()
    
    for param, default_val in default_values.items():
        if param != dim:  # Don't filter on the dimension we're sweeping
            # Use a small tolerance for floating point comparison
            if param in ['vd', 'large_move_penalty', 'large_turn_penalty']:
                filtered_df = filtered_df[abs(filtered_df[param] - default_val) < 1e-6]
            else:  # For integer parameters like walkerbots
                filtered_df = filtered_df[filtered_df[param] == default_val]
    
    print(f"\n=== Filtering for {dim} sweep ===")
    print(f"Original data: {len(df_runs)} runs")
    print(f"After filtering: {len(filtered_df)} runs")
    
    if len(filtered_df) == 0:
        print(f"Warning: No runs found with default values for {dim} sweep!")
        continue
    
    grouped = (
        filtered_df.groupby(dim)[metrics]
        .agg(['mean', 'std', 'sem', 'count'])
        .reset_index()
        .sort_values(dim)
    )
    summary_tables[dim] = grouped
    print(f"\n=== Summary by {dim} ===")
    print(grouped.to_string(index=False))

In [ ]:
# # Violin/Box plots

# PLOT_STYLE = ["violin", "box"][1]
# for metric in metrics:
#     fig, axes = plt.subplots(1, 4, figsize=FIG_SIZE, sharey=True)
#     for i, dim in enumerate(sweep_dimensions):
#         ax = axes[i]
#         # Treat each distinct value of the sweep dim as a category (sorted for nice ordering)
#         x_order = sorted(df_runs[dim].dropna().unique())

#         if PLOT_STYLE == "violin":
#             # Violin plots with quartile lines; no error bars
#             sns.violinplot(
#                 data=df_runs, x=dim, y=metric,
#                 order=x_order, cut=0, inner="quartile", scale="width",
#                 ax=ax
#             )
#             # Overlay raw points
#             sns.stripplot(
#                 data=df_runs, x=dim, y=metric,
#                 order=x_order, jitter=0.15, alpha=0.5, size=3,
#                 ax=ax, color="k"  # black points; remove color kwarg if you prefer default palette
#             )

#         elif PLOT_STYLE == "box":
#             # Box-plots (median & IQR), hide outliers (we'll show raw points instead)
#             sns.boxplot(
#                 data=df_runs, x=dim, y=metric,
#                 order=x_order, showfliers=False,
#                 ax=ax
#             )
#             # Overlay raw points
#             sns.stripplot(
#                 data=df_runs, x=dim, y=metric,
#                 order=x_order, jitter=0.15, alpha=0.5, size=3,
#                 ax=ax, color="k"
#             )

#         # Common
#         # ax.set_title(dim)
#         # ax.set_xlabel(dim)
#         ax.set_xlabel(OTHER_METADATA[dim]["name"])
#         if i == 0:
#             # ax.set_ylabel(metric)
#             ax.set_ylabel(OTHER_METADATA[metric]["name"])
#         else:
#             ax.set_ylabel("")
#     plt.tight_layout()
#     fname = f"data_sweep/sweep_{metric}_{PLOT_STYLE}.png"
#     plt.savefig(fname, dpi=300)
#     print(f"Saved: {fname}")
#     plt.show()

# Violin/Box plots

PLOT_STYLE = ["violin", "box"][1]
for metric in metrics:
    fig, axes = plt.subplots(1, 4, figsize=FIG_SIZE, sharey=True)
    for i, dim in enumerate(sweep_dimensions):
        ax = axes[i]
        
        # ADD THIS: Filter data for this dimension to use default values for other params
        filtered_df = df_runs.copy()
        for param, default_val in default_values.items():
            if param != dim:
                filtered_df = filtered_df[filtered_df[param] == default_val]
        
        # Treat each distinct value of the sweep dim as a category (sorted for nice ordering)
        x_order = sorted(filtered_df[dim].dropna().unique())  # CHANGED: use filtered_df

        if PLOT_STYLE == "violin":
            # Violin plots with quartile lines; no error bars
            sns.violinplot(
                data=filtered_df, x=dim, y=metric,  # CHANGED: use filtered_df
                order=x_order, cut=0, inner="quartile", scale="width",
                ax=ax
            )
            # Overlay raw points
            sns.stripplot(
                data=filtered_df, x=dim, y=metric,  # CHANGED: use filtered_df
                order=x_order, jitter=0.15, alpha=0.5, size=3,
                ax=ax, color="k"  # black points; remove color kwarg if you prefer default palette
            )

        elif PLOT_STYLE == "box":
            # Box-plots (median & IQR), hide outliers (we'll show raw points instead)
            sns.boxplot(
                data=filtered_df, x=dim, y=metric,  # CHANGED: use filtered_df
                order=x_order, showfliers=False,
                ax=ax
            )
            # Overlay raw points
            sns.stripplot(
                data=filtered_df, x=dim, y=metric,  # CHANGED: use filtered_df
                order=x_order, jitter=0.15, alpha=0.5, size=3,
                ax=ax, color="k"
            )

        # Common
        # ax.set_title(dim)
        # ax.set_xlabel(dim)
        ax.set_xlabel(OTHER_METADATA[dim]["name"])
        if i == 0:
            # ax.set_ylabel(metric)
            ax.set_ylabel(OTHER_METADATA[metric]["name"])
        else:
            ax.set_ylabel("")
    plt.tight_layout()
    fname = f"data_sweep/sweep_{metric}_{PLOT_STYLE}.png"
    plt.savefig(fname, dpi=300)
    print(f"Saved: {fname}")
    plt.show()

## First select best run per setting, then plot trends


In [ ]:

best_runs = df_runs.loc[df_runs.groupby(sweep_dimensions)['eating_events_per_episode'].idxmax()]
for metric in metrics:
    fig, axes = plt.subplots(1, 4, figsize=FIG_SIZE, sharey=True)
    for i, dim in enumerate(sweep_dimensions):
        sns.pointplot(
            data=best_runs, x=dim, y=metric, 
            errorbar="sd",
            ax=axes[i], capsize=0.1
        )
        # axes[i].set_title(f"Best Runs: {metric} vs {dim}")
    plt.tight_layout()
    plt.show()


In [ ]:
# for metric in metrics:
#     for fixed in sweep_dimensions:
#         sub_dims = [d for d in sweep_dimensions if d != fixed]
#         pivot = (
#             df_runs.groupby(sub_dims)[metric]
#             .mean().reset_index()
#             .pivot(index=sub_dims[0], columns=sub_dims[1], values=metric)
#         )
#         sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis")
#         plt.title(f"{metric}, averaged over {fixed}")
#         plt.show()

# sns.catplot(
#     data=df_runs,
#     x="vd", y="eating_events_per_episode",
#     hue="large_move_penalty",
#     col="large_turn_penalty",
#     kind="bar", ci="sd", col_wrap=3, height=3
# )
# plt.show()


In [ ]:
df_runs[ df_runs["run"] == 0 ].shape

In [ ]:
best_runs